<a href="https://colab.research.google.com/github/efarfan-dev/LaLiga_Poisson_Model/blob/main/LaLiga_MatchPredictor_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn
from scipy.stats import poisson, skellam
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
df = pd.read_csv("https://www.football-data.co.uk/mmz4281/2526/SP1.csv")
df = df[['HomeTeam','AwayTeam','FTHG','FTAG']]
df = df.rename(columns = {'FTHG': 'HomeGoals', 'FTAG': 'AwayGoals'})
df.head()


In [ ]:
goal_model_data = pd.concat([df[['HomeTeam','AwayTeam','HomeGoals']].assign(home = 1).rename(columns={'HomeTeam':'team','AwayTeam':'opponent', 'HomeGoals':'goals'}),
                             df[['AwayTeam','HomeTeam','AwayGoals']].assign(home = 0).rename(columns = {'AwayTeam': 'team', 'HomeTeam':'opponent','AwayGoals':'goals'})])

In [ ]:
poisson_model = smf.glm(formula = "goals ~ home + team + opponent", data = goal_model_data, family = sm.families.Poisson()).fit()
poisson_model.summary()

Expected mean number of goals

In [ ]:
home_team = 'Real Madrid'
away_team = 'Alaves'

In [ ]:
home_score_rate = poisson_model.predict(pd.DataFrame(data = {'team':home_team, 'opponent': away_team, 'home': 1}, index=[1]))
away_score_rate = poisson_model.predict(pd.DataFrame(data = {'team':away_team, 'opponent': home_team, 'home':0}, index=[1]))
print(home_team + ' against ' + away_team + ' expect to score ' + str(home_score_rate))
print(away_team  + ' against ' + home_team + ' expect to score ' + str(away_score_rate))


Match Simulation

In [ ]:
def simulate_match(foot_model, homeTeam, awayTeam, max_goals = 10):
  home_goals_avg = foot_model.predict(pd.DataFrame(data = {'team': homeTeam,
                                                           'opponent':awayTeam, 'home':1},
                                                   index=[1])).values[0]

  away_goals_avg = foot_model.predict(pd.DataFrame(data = {'team': awayTeam,
                                                           'opponent':homeTeam, 'home':0},
                                                   index=[1])).values[0]
  team_pred = [[poisson.pmf(i, team_avg) for i in range(0, max_goals+1)] for team_avg in [home_goals_avg, away_goals_avg]]
  return (np.outer(np.array(team_pred[0]), np.array(team_pred[1])))

In [ ]:
max_goals = 5

score_matrix=simulate_match(poisson_model, home_team, away_team, max_goals)
score_matrix

In [ ]:
import seaborn as sns

ax = sns.heatmap(score_matrix, linewidth=0.8, annot=True)
ax.set_xlabel("Goals scored by " + away_team)
ax.set_ylabel("Goals scored by " + home_team)
plt.show()

In [ ]:
#Home, Draw, Away Probabilities
homewin = np.sum(np.tril(score_matrix, -1))
draw = np.sum(np.diag(score_matrix))
awaywin = np.sum(np.triu(score_matrix, 1))

print(home_team + ' win probability: ' + str(round(homewin*100,2)) + '%')
print(away_team + ' win probability: ' + str(round(awaywin*100,2)) + '%')
print('Draw probability: ' + str(round(draw*100,2)) + '%')

print('---------------------------------------')

round(homewin, 5), round(draw, 5), round(awaywin, 5)